In [33]:
import pandas as pd
import os
path_carpeta = "/Users/l.antonelli/Documents/proyect-ing/Inscriptos"



In [34]:
def procesar_excel_inscriptos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.iloc[3:]
    df = df.iloc[:-1]
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    print(df.columns)
    return df

def validar_columnas_excel(df: pd.DataFrame):
    if df.columns.is_unique:
        print("Columnas únicas")
    else:
        raise ValueError(f"Columnas no únicas {df.columns} para df \n{df}")
    

In [35]:
dfs = []

# Iterate over all files in the directory
for filename in os.listdir(path_carpeta):
    if filename.endswith(".xlsx") and not filename.startswith("~$"):
        file_path = os.path.join(path_carpeta, filename)
        
        # Remove extension and split by the first hyphen
        # e.g., "2022-1C" -> ["2022", "1C"]
        name_without_ext = os.path.splitext(filename)[0]
        parts = name_without_ext.split('-', 1)
        
        if len(parts) >= 2:
            year = parts[0].strip()
            period = parts[1].strip()
            
            try:
                # Read the Excel file
                df = pd.read_excel(file_path)
                df = procesar_excel_inscriptos(df)
                validar_columnas_excel(df)
                
                # Add the two new columns
                df['Year'] = year
                df['Period'] = period
                
                dfs.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Consolidate all DataFrames
if dfs:
    final_df = pd.concat(dfs, axis=0, ignore_index=True)
    print(f"Successfully consolidated {len(dfs)} files.")
    print(final_df.head())
else:
    print("No files found to consolidate.")

Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Cant. Inscriptos', 'Propuestas'],
      dtype='object', name=3)
Columnas únicas
Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Cant. Inscriptos', 'Propuestas'],
      dtype='object', name=3)
Columnas únicas
Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Cant. Inscriptos', 'Propuestas'],
      dtype='object', name=3)
Columnas únicas
Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Cant. Inscriptos', 'Propuestas'],
      dtype='object', name=3)
Columnas únicas
Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Cant. Inscriptos', 'Propuestas'],
      dtype='object', name=3)
Columnas únicas
Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Cant. Inscriptos', 'Propuestas'],
      dtype='object', name=3)
Columnas únicas
Index(['Código', 'Actividad', 'Comisión', 'Turno', 'Cupo Máximo',
       'Ca

In [36]:
final_df

3,Código,Actividad,Comisión,Turno,Cupo Máximo,Cant. Inscriptos,Propuestas,Year,Period
0,G10,Agrología e Información Rural,Agr Inf Rural,Tarde,NaN,19,Agrimensura,2023,2C
1,IA25,Algebra,Álgebra C1,Noche,50,48,Tecnicatura Universitaria en Inteligencia Arti...,2023,2C
2,IA25,Algebra,Álgebra C2,Noche,50,48,Tecnicatura Universitaria en Inteligencia Arti...,2023,2C
3,FB9,Algebra lineal,110 ALG YLINEAL,Mañana,102,0,Ingeniería Industrial,2023,2C
4,FB9,Algebra lineal,110 ALG YLINEAL,Mañana,102,0,Ingeniería Electrónica,2023,2C
...,...,...,...,...,...,...,...,...,...
10113,C23,Transporte II,Transporte II,NaN,NaN,33,Ingeniería Civil,2025,1C
10114,C27,Transporte III,Transport III,Mañana,NaN,30,Ingeniería Civil,2025,1C
10115,G2,Trigonometría y elementos de topografía,Trig y Elm Trig,NaN,NaN,13,Agrimensura,2025,1C
10116,L1816,Variedades diferenciables,Varied Diferenc,Mañana,NaN,4,Licenciatura en Matemática,2025,1C


In [37]:
import unicodedata

final_df.columns = [
    unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8').lower().replace(' ', '_')
    for col in final_df.columns
]

final_df.columns

Index(['codigo', 'actividad', 'comision', 'turno', 'cupo_maximo',
       'cant._inscriptos', 'propuestas', 'year', 'period'],
      dtype='object')

In [38]:
final_df.describe()

,codigo,actividad,comision,turno,cupo_maximo,cant._inscriptos,propuestas,year,period
count,10118,10118,10118,6223,6647,10118,10118,10118,10118
unique,507,453,1898,3,108,160,16,4,4
top,CI2401,Introducción a la Matemática,LAS PAREJAS,Tarde,0,0,Ingeniería Electrónica,2025,1C
freq,528,528,489,2945,930,1139,1412,3274,4879


In [39]:
# Quitar columna carrera y turno que no aporta nada (no da los inscriptos por carrera en cada comision)
final_df = final_df.drop(columns=['propuestas', 'turno'])

In [40]:
# Quitar duplicados
final_df = final_df.drop_duplicates()

In [41]:
# Partir las materias anuales en 1C y 2C
final_df['period'] = final_df['period'].apply(lambda x: ['1C', '2C'] if x == 'Anual' else x)
final_df = final_df.explode('period', ignore_index=True).reset_index(drop=True)
final_df['period'].replace(to_replace='Curso de Nivelación', value = '2C', inplace = True)

/var/folders/8b/yqrbg37544b3d_vg48ysd0pm0000gq/T/ipykernel_43684/1765113237.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df['period'].replace(to_replace='Curso de Nivelación', value = '2C', inplace = True)


In [42]:
final_df.to_excel('final_df.xlsx', index=False)